# Качество данных — демонстрация

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
DATA = Path("../data")
LOCAL = Path("../data")
assert (DATA / "decisions.csv").is_file()
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__)
print(decisions.shape)

## B02 — слайд 2: Знакомая таблица HMDA

In [ ]:
decisions.head(4)

## B03 — слайд 3: Словарь полей

In [ ]:
print(decisions.columns.tolist())
print(decisions.dtypes)

## B04 — слайд 4: Правила для нашего расчёта

In [ ]:
approved = decisions["action_taken"].isin([1, 2]).sum()
n = len(decisions)
print(approved, n, f"{100 * approved / n:.2f}%")

## B05 — слайд 5: Профиль пропусков

In [ ]:
decisions.info()

In [ ]:
print(decisions.isna().sum())
print(decisions["action_taken"].value_counts(dropna=False))

In [ ]:
unknown_income = decisions["income"].isna()
decisions.loc[unknown_income, ["source_row", "action_taken", "income"]].head(5)

## B06 — слайд 6: Пропуск и ноль

In [ ]:
illustration = pd.Series([100.0, 200.0, float("nan")])
print(illustration)
print(illustration.mean())
print(illustration.fillna(0).mean())

## B07 — слайд 7: Доля пропусков по исходам

In [ ]:
missing_input = decisions.copy()
missing_input["income_missing"] = missing_input["income"].isna()

In [ ]:
missing_profile = missing_input.groupby("action_taken").agg(
    n=("income", "size"),
    missing=("income_missing", "sum"),
).reset_index()
missing_profile["missing_rate"] = missing_profile["missing"] / missing_profile["n"]
missing_profile

## B08 — слайд 8: Удаление строк с неизвестным доходом

In [ ]:
complete_income = decisions.dropna(subset=["income"])
print(len(decisions), len(complete_income))

In [ ]:
for label, part in [("Все решения", decisions), ("Известный доход", complete_income)]:
    k = part["action_taken"].isin([1, 2]).sum()
    print(label, len(part), k, k / len(part))

## B10 — слайд 10: Учебная копия выгрузки

In [ ]:
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
export.head(5)

In [ ]:
print(export.shape)
print(export.dtypes)
raw_copy = export.copy(deep=True)

## B11 — слайд 11: Текст в число и ошибки преобразования

In [ ]:
numeric_amount = pd.to_numeric(export["loan_amount"], errors="coerce")
parse_error = export["loan_amount"].notna() & numeric_amount.isna()
print(export.loc[parse_error, ["source_row", "loan_amount"]])

## B12 — слайд 12: Пробелы и коды категорий

In [ ]:
normalized_county = export["county_code"].str.strip()
print(export.loc[export["county_code"].ne(normalized_county), ["source_row", "county_code"]])

In [ ]:
outcome_num = pd.to_numeric(export["action_taken"], errors="coerce")
invalid_outcome = ~outcome_num.isin([1, 2, 3])
print(export.loc[invalid_outcome, ["source_row", "action_taken"]])

## B13 — слайд 13: Единицы измерения

In [ ]:
prepared = decisions.copy()
prepared["income_usd"] = prepared["income"] * 1000
prepared[["income", "income_usd"]].head(4)

## B14 — слайд 14: Повторы организации и строки

In [ ]:
print("Повторы LEI:", decisions["lei"].duplicated().sum())
print("Повторы технического ключа:", decisions["source_row"].duplicated().sum())

In [ ]:
business = decisions.drop(columns="source_row")
print(business.duplicated().sum())
assert business.duplicated().sum() == 73

In [ ]:
pairs = business.duplicated(keep=False)
decisions.loc[pairs].head(6)

## B15 — слайд 15: Подтверждённые повторы доставки

In [ ]:
known_key = export["source_row"].notna()
repeat_mask = known_key & export["source_row"].duplicated(keep=False)
export.loc[repeat_mask]

In [ ]:
repeated = export.loc[known_key].drop_duplicates()
assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
replay_mask = known_key & export["source_row"].duplicated()
unique_export = export.loc[~replay_mask].copy()
print(len(export), len(unique_export))

## B16 — слайд 16: Необычно большая сумма

In [ ]:
amount = decisions["loan_amount"]
q1, q3 = amount.quantile([.25, .75])
iqr = q3 - q1
upper_flag = q3 + 1.5 * iqr
print(q1, q3, upper_flag)

In [ ]:
flagged = decisions.loc[amount.gt(upper_flag)]
print("Кандидатов на разбор:", len(flagged))
flagged[["source_row", "loan_amount", "action_taken"]].head(5)

## B17 — слайд 17: Среднее и медиана суммы

In [ ]:
print("Исходный mean/median:", amount.mean(), amount.median())
without_tail = decisions.loc[amount.le(upper_flag)]
print("Без флага хвоста:", len(without_tail), without_tail["loan_amount"].mean())

## B18 — слайд 18: Пересечение условий

In [ ]:
missing_mask = decisions["income"].isna()
denial_mask = decisions["action_taken"].eq(3)
print(missing_mask.sum(), denial_mask.sum())
print((missing_mask | denial_mask).sum(), (missing_mask & denial_mask).sum())

## B20 — слайд 20: Функция очистки

In [ ]:
def clean_export(source):
    out = source.copy(deep=True)
    known_key = out["source_row"].notna()
    repeated = out.loc[known_key].drop_duplicates()
    assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
    replay_mask = known_key & out["source_row"].duplicated()
    replay_n = int(replay_mask.sum())
    out = out.loc[~replay_mask].copy()
    out["county_code"] = out["county_code"].str.strip()
    parse_counts = {}
    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(out[field], errors="coerce")
        parse_counts[field] = int((out[field].notna() & numbers.isna()).sum())
        out[field] = numbers
    reason = pd.Series("", index=out.index, dtype="string")
    reason.loc[out["source_row"].isna()] += "unknown_source_row;"
    bad_amount = out["loan_amount"].isna() | out["loan_amount"].le(0)
    reason.loc[bad_amount] += "invalid_loan_amount;"
    reason.loc[~out["action_taken"].isin([1, 2, 3])] += "invalid_action_taken;"
    bad_county = ~out["county_code"].eq("11001").fillna(False)
    reason.loc[bad_county] += "invalid_county;"
    invalid = reason.ne("")
    clean = out.loc[~invalid].copy()
    quarantine = source.loc[out.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]
    audit = {
        "input": len(source), "replays": replay_n,
        "clean": len(clean), "quarantine": len(quarantine),
        "parse_errors": parse_counts,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit

In [ ]:
clean, quarantine, audit = clean_export(export)
print(audit)
quarantine

## B21 — слайд 21: Проверки результата и входа

In [ ]:
from pandas.testing import assert_frame_equal
assert_frame_equal(export, raw_copy)
assert clean["source_row"].is_unique
assert clean["action_taken"].isin([1, 2, 3]).all()
assert clean["loan_amount"].gt(0).all()
assert len(export) == len(clean) + len(quarantine) + audit["replays"]

In [ ]:
again, quarantine_again, audit_again = clean_export(export)
assert_frame_equal(clean, again)
assert_frame_equal(quarantine, quarantine_again)
assert audit == audit_again

## B22 — слайд 22: Показатель после очистки

In [ ]:
def show_approval(label, table):
    n_rows = len(table)
    assert n_rows > 0
    n_approved = table["action_taken"].isin([1, 2]).sum()
    print(label, n_approved, n_rows, f"{100 * n_approved / n_rows:.2f}%")

show_approval("Оригинал", decisions)
show_approval("После учебной очистки", clean)
complete_clean = clean.dropna(subset=["income"])
show_approval("Известный доход после очистки", complete_clean)